# Periodos orbitales y rotacionales de todos los sistemas (v2)

Versión corregida de `explore_priors.ipynb`. Cambios:

- **Catálogo**: lee `Resultados_Orion/priors_v2.csv` (sale de `1b_profundidades_eclipse_v2.ipynb`)
  en lugar del `priors.csv` local. Ahí van ya:
  - los **periodos orbitales corregidos** con la prueba par-impar por sistema (§3 de `1b`):
    199780530, 249074181, 427347964 y 427395536 tienen en realidad el **doble** del periodo que daba
    BLS (dos eclipses parecidos encimados), y 176775425 S98 y 264741359 S32 estaban al doble del
    periodo del sistema;
  - **P_rot recalculado** con la efeméride corregida y los eclipses enmascarados en su fase
    real, con su incertidumbre y los 3 picos principales del periodograma de cada sector.
- **Índice interno**: cada sistema se identifica con un número del 1 al N, ordenado por periodo
  orbital. La correspondencia con el TIC está en la tabla de la sección 1 y en
  `figuras_explore_priors/indice_sistemas.csv`.
- **Una fila por sistema** en las gráficas de síntesis. Cada punto lleva barras de error, y la
  dispersión entre sectores se muestra aparte.
- **Periodo de rotación de consenso** entre sectores (sección 2), porque el pico máximo de
  Lomb-Scargle a veces salta entre armónicos de un sector a otro.
- **Excentricidad en la barra de color**: $|e\cos\omega|$, medido con la fase del eclipse
  secundario. Es la parte robusta de la excentricidad y una cota inferior de $e$.

**Salidas**: todas las figuras (PNG, 200 dpi) y las tablas `indice_sistemas.csv` y `tiempos_marea.csv`
se guardan en `Resultados_Orion/figuras_explore_priors/` (variable `DIR_FIG`).

Colores: **cadencia del sector** (azul = 30 min, S5/S6; naranja = 10 min, S32/S33;
aqua = 200 s–6.7 min, S87/S98) y **$|e\cos\omega|$** (rampa azul). Los sistemas sin secundario
detectado se dibujan huecos en gris.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, LogLocator, MaxNLocator

RUTA_ORION = Path("/home/luciam/Documents/starry/Resultados_Orion")
PATH_PRIORS = RUTA_ORION / "priors_v2.csv"
DIR_FIG = RUTA_ORION / "figuras_explore_priors"   # todas las figuras y tablas de este notebook
DIR_FIG.mkdir(parents=True, exist_ok=True)

TOL_CONSENSO = 0.05   # dos picos de sectores distintos "coinciden" si difieren menos de 5 %

# -------- estilo --------
TINTA, TINTA_2, GRIS = "#0b0b0b", "#52514e", "#a3a29c"
CADENCIA = {"30 min (S5, S6)": "#2a78d6", "10 min (S32, S33)": "#eb6834", "≤ 6.7 min (S87, S98)": "#1baf7a"}
RAMPA_E = LinearSegmentedColormap.from_list("azul", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11, "axes.edgecolor": TINTA_2,
    "axes.labelcolor": TINTA, "xtick.color": TINTA_2, "ytick.color": TINTA_2,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
    "figure.facecolor": "white", "axes.facecolor": "white",
})


DESPLAZ = {"30 min (S5, S6)": -0.22, "10 min (S32, S33)": 0.0, "≤ 6.7 min (S87, S98)": 0.22}


def clase_cadencia(sector):
    if sector in (5, 6):
        return "30 min (S5, S6)"
    if sector in (32, 33):
        return "10 min (S32, S33)"
    return "≤ 6.7 min (S87, S98)"


def etiquetar(ax, xs, ys, textos, sep_px=13):
    # Coloca cada etiqueta en la primera posición libre (lejos de otros puntos y etiquetas ya puestas).
    ax.figure.canvas.draw()
    pts = ax.transData.transform(np.column_stack([xs, ys]))
    ocupado = [tuple(p) for p in pts]
    opciones = [(7, 6), (7, -13), (-9, 6), (-9, -13), (0, 11), (0, -17), (13, -3), (-15, -3)]
    for (px, py), txt in zip(pts, textos):
        for dx, dy in opciones:
            cx, cy = px + dx + (4 if dx >= 0 else -4), py + dy + 4
            if all(np.hypot(cx - ox, cy - oy) > sep_px for ox, oy in ocupado):
                break
        ax.annotate(txt, ax.transData.inverted().transform((px, py)), xytext=(dx, dy),
                    textcoords="offset points", fontsize=8.5, color=TINTA,
                    ha="left" if dx >= 0 else "right", zorder=6)
        ocupado.append((cx, cy))


def dibujar_ecosw(ax, x, r, color, piso, ok=True, z=3, s=75):
    # Excéntrica (>= 3 sigma): punto con barra de error. Consistente con 0: límite superior al 95 % (flecha).
    # Sin secundario detectado: triángulo en la franja inferior.
    a = 1 if ok else 0.6
    if not np.isfinite(r["abs_ecosw"]):
        ax.scatter(x, piso, s=s * 0.8, marker="v", facecolor=color, edgecolor=TINTA_2, lw=1.1, alpha=a, zorder=z)
        return piso
    if r["excentrica"]:
        ax.errorbar(x, r["abs_ecosw"], yerr=r["ecosw_sis_err"], fmt="none", ecolor=GRIS, elinewidth=1, zorder=z - 1)
        ax.scatter(x, r["abs_ecosw"], s=s, marker="o" if ok else "D", facecolor=color, edgecolor=TINTA_2, lw=1.1,
                   alpha=a, zorder=z)
        return r["abs_ecosw"]
    y = r["ecosw_lim95"]
    ax.errorbar(x, y, yerr=y * 0.45, uplims=True, fmt="none", ecolor=TINTA_2, elinewidth=1, capsize=3,
                alpha=a, zorder=z - 1)
    ax.scatter(x, y, s=s * 0.75, marker="_", color=TINTA_2, lw=2, alpha=a, zorder=z)
    ax.scatter(x, y, s=s * 0.55, marker="o" if ok else "D", facecolor=color, edgecolor=TINTA_2, lw=0.9,
               alpha=a, zorder=z + 1)
    return y


def eje_log(ax, eje="both"):
    fmt = FuncFormatter(lambda v, _: f"{v:g}")
    for a in ([ax.xaxis, ax.yaxis] if eje == "both" else [ax.xaxis if eje == "x" else ax.yaxis]):
        a.set_major_locator(LogLocator(base=10, subs=(1, 2, 5)))
        a.set_major_formatter(fmt)
        a.set_minor_formatter(FuncFormatter(lambda v, _: ""))

## 1) Catálogo e índice de sistemas

Por cada TIC se toman los sectores **válidos**. Se descartan los marcados `eclipse_ancho`
(BLS no encontró un eclipse separado) cuando el TIC tiene otros sectores buenos. Si todos sus
sectores tienen esa marca, se conservan y el sistema se marca `revisar`.

$P_{\rm orb}$ del sistema es la mediana de sus sectores válidos. Su incertidumbre es la
dispersión entre sectores (semirrango), o la resolución de BLS si solo hay un sector.

$e\cos\omega$ del sistema es el promedio pesado de $\tfrac{\pi}{2}[(c_2-c_1)-\tfrac12]$ en los sectores con
secundario detectado, con el error de bootstrap de cada sector (§2.10 de `1b`). Si los sectores no
concuerdan ($\chi^2_{\rm red}>1$), el error combinado se multiplica por $\sqrt{\chi^2_{\rm red}}$. Un sistema
se considera **excéntrico** si $|e\cos\omega|/\sigma\ge3$. Si no, se reporta el límite superior
$|e\cos\omega|_{95}=|e\cos\omega|+1.96\,\sigma$.

In [ ]:
df = pd.read_csv(PATH_PRIORS)
df["cadencia"] = df["Sector"].map(clase_cadencia)
df["ancho"] = df["Calidad_depth"].fillna("").str.contains("eclipse_ancho")

# sectores válidos por TIC
def validos(g):
    buenos = g.loc[~g["ancho"]]
    return buenos if len(buenos) else g

val = df.groupby("TIC", group_keys=False).apply(validos)

sist = val.groupby("TIC").agg(
    P_orb=("P_orb", "median"),
    P_orb_min=("P_orb", "min"), P_orb_max=("P_orb", "max"),
    n_sectores=("Sector", "nunique"),
    sectores=("Sector", lambda s: ",".join(str(int(x)) for x in sorted(s))),
    revisar=("ancho", "all"),
    P_corregido=("P_corregido", "any"),
).reset_index()
sist["P_orb_err"] = np.where(sist["n_sectores"] > 1, (sist["P_orb_max"] - sist["P_orb_min"]) / 2, np.nan)

# e cos w del sistema: promedio pesado de los sectores con secundario detectado (error de bootstrap, §2.10 de
# 1b). Si los sectores no concuerdan (chi2_red > 1), el error se infla por sqrt(chi2_red): eso absorbe los
# sistemáticos que el bootstrap no ve (manchas que desplazan mínimos, residuos del detrending).
def combinar_ecosw(g):
    g = g.loc[g["secundario_detectado"].astype(bool) & g["ecosw_err"].notna() & (g["ecosw_err"] > 0)]
    if g.empty:
        return pd.Series({"ecosw_sis": np.nan, "ecosw_sis_err": np.nan, "ecosw_chi2red": np.nan, "n_ecosw": 0})
    w = 1 / g["ecosw_err"] ** 2
    m = float((w * g["ecosw"]).sum() / w.sum())
    e = float(w.sum() ** -0.5)
    n = len(g)
    chi2 = float((w * (g["ecosw"] - m) ** 2).sum() / (n - 1)) if n > 1 else np.nan
    if n > 1 and chi2 > 1:
        e *= np.sqrt(chi2)
    return pd.Series({"ecosw_sis": m, "ecosw_sis_err": e, "ecosw_chi2red": chi2, "n_ecosw": n})


ecs = val.groupby("TIC").apply(combinar_ecosw)
sist = sist.drop(columns=[c for c in ecs.columns if c in sist]).merge(ecs, left_on="TIC", right_index=True, how="left")
sist["abs_ecosw"] = sist["ecosw_sis"].abs()
sist["signif_ecosw"] = sist["abs_ecosw"] / sist["ecosw_sis_err"]
sist["ecosw_lim95"] = sist["abs_ecosw"] + 1.96 * sist["ecosw_sis_err"]   # límite superior de |e cos w|
sist["excentrica"] = sist["signif_ecosw"] >= 3

# índice interno: 1..N ordenado por periodo orbital
sist = sist.sort_values("P_orb").reset_index(drop=True)
sist.insert(0, "indice", np.arange(1, len(sist) + 1))
IDX = dict(zip(sist["TIC"], sist["indice"]))
df["indice"] = df["TIC"].map(IDX)
val = val.assign(indice=val["TIC"].map(IDX))

print(f"{len(df)} filas (TIC, sector), {len(sist)} sistemas; "
      f"{int(sist['P_corregido'].sum())} con algún P_orb corregido en 1b; "
      f"{int(sist['revisar'].sum())} marcados 'revisar' (sin eclipse separado)")
sist[["indice", "TIC", "P_orb", "P_orb_err", "n_sectores", "sectores", "ecosw_sis", "ecosw_sis_err", "signif_ecosw",
      "ecosw_lim95", "excentrica", "P_corregido", "revisar"]].round(5)

## 2) Periodo de rotación de consenso

En cada sector, Lomb-Scargle da varios picos, y el más alto a veces es un armónico ($P/2$ o
$2P$) del periodo real: con dos manchas en lados opuestos, la curva se repite dos veces por
rotación. El pico dominante cambia de sector a sector según cómo evolucionen las manchas.

**Regla de consenso.** Se juntan los 3 picos principales de cada sector válido. Para cada
candidato $c$ se cuenta en cuántos sectores hay un pico a menos de 5 % de $c$, y se suma la
potencia de esos picos. Se elige el candidato con **más sectores de acuerdo** y, en caso de
empate, el de **mayor potencia total**. Entonces:

- $P_{\rm rot}$ es la mediana de los picos que coinciden;
- su error es la mitad del rango de esos picos, o el error por ancho de pico de `1b` si solo
  hay un sector;
- el sistema se marca `ambiguo` si el periodo elegido no aparece en todos sus sectores.

**Relación con la órbita.** Se reporta $P_{\rm rot}/P_{\rm orb}$ y se marca si cae a menos de 3 % de
½, 1 o 2. Recuerda que $P_{\rm rot}=P_{\rm orb}$ puede ser sincronización **o** reflexión, y que
$P_{\rm orb}/2$ puede ser variación elipsoidal (ver §4b de `1b`).

In [ ]:
def picos_sector(fila):
    if not isinstance(fila["P_rot_picos"], str) or not fila["P_rot_picos"]:
        return []
    P = [float(x) for x in fila["P_rot_picos"].split(";")]
    w = [float(x) for x in fila["P_rot_picos_pot"].split(";")]
    return list(zip(P, w))


def consenso(g):
    picos = {int(r["Sector"]): picos_sector(r) for _, r in g.iterrows()}
    picos = {s: p for s, p in picos.items() if p}
    if not picos:
        return pd.Series({"P_rot": np.nan, "P_rot_err": np.nan, "n_acuerdo": 0, "ambiguo": True})
    mejor = None
    for c in sorted({P for p in picos.values() for P, _ in p}):
        coincide = {}
        for s, p in picos.items():
            cerca = [(P, w) for P, w in p if abs(P / c - 1) < TOL_CONSENSO]
            if cerca:
                coincide[s] = max(cerca, key=lambda x: x[1])
        clave = (len(coincide), sum(w for _, w in coincide.values()))
        if mejor is None or clave > mejor[0]:
            mejor = (clave, coincide)
    (n, _), coincide = mejor
    Ps = np.array([P for P, _ in coincide.values()])
    if len(Ps) > 1:
        err = (Ps.max() - Ps.min()) / 2
    else:
        s = next(iter(coincide))
        r = g.loc[g["Sector"] == s].iloc[0]
        err = r["P_rot_err"] * (Ps[0] / r["P_rot"]) ** 2 if np.isfinite(r["P_rot"]) else np.nan
    return pd.Series({"P_rot": float(np.median(Ps)), "P_rot_err": float(err), "n_acuerdo": n,
                      "ambiguo": n < len(picos)})


rot = val.groupby("TIC").apply(consenso).reset_index()
sist = sist.drop(columns=[c for c in ["P_rot", "P_rot_err", "n_acuerdo", "ambiguo"] if c in sist]).merge(rot, on="TIC")
sist["cociente"] = sist["P_rot"] / sist["P_orb"]
sist["armonico"] = [next((f"{k:g}" for k in (0.5, 1, 2) if abs(q / k - 1) < 0.03), "") if np.isfinite(q) else ""
                    for q in sist["cociente"]]

tabla = sist[["indice", "TIC", "n_sectores", "P_orb", "P_orb_err", "P_rot", "P_rot_err", "n_acuerdo", "ambiguo",
              "cociente", "armonico", "ecosw_sis", "ecosw_sis_err", "signif_ecosw", "ecosw_lim95", "excentrica",
              "abs_ecosw", "P_corregido", "revisar"]]
tabla.to_csv(DIR_FIG / "indice_sistemas.csv", index=False)
print(f"Guardado: {DIR_FIG / 'indice_sistemas.csv'}")
tabla.round(4)

## 3) Periodo orbital por sector

Cada columna es un sistema (índice interno). Los puntos son los sectores, coloreados por
cadencia y ligeramente desplazados a los lados según la cadencia para que no se tapen. Si el sistema es estable deberían quedar uno encima del otro. Las **✕ grises** son
los valores de v1 que `1b` corrigió, unidas con una línea al valor corregido.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
for _, r in df.sort_values("indice").iterrows():
    x = r["indice"] + DESPLAZ[r["cadencia"]]
    if r["P_corregido"]:
        ax.plot([x, x], [r["P_orb_v1"], r["P_orb"]], color=GRIS, lw=1, zorder=1)
        ax.scatter(x, r["P_orb_v1"], marker="x", s=45, color=GRIS, lw=1.5, zorder=2)
    hueco = r["ancho"]
    c = CADENCIA[r["cadencia"]]
    ax.scatter(x, r["P_orb"], s=55, zorder=3, facecolor="white" if hueco else c, edgecolor=c, lw=1.5)

ax.set_yscale("log"); eje_log(ax, "y")
ax.set_xticks(sist["indice"])
ax.set_xlim(0.3, len(sist) + 0.7)
ax.set_xlabel("Sistema (índice interno)")
ax.set_ylabel(r"$P_{\rm orb}$ (d)")
ax.set_title("Periodo orbital de cada sector, por sistema")
ax.grid(axis="y", which="major", alpha=0.25)
ax.grid(axis="x", alpha=0.12)
leyenda = [Line2D([], [], ls="", marker="o", ms=8, color=c, label=k) for k, c in CADENCIA.items()]
leyenda += [Line2D([], [], ls="", marker="o", ms=8, mfc="white", mec=TINTA_2, label="sin eclipse separado"),
            Line2D([], [], ls="-", marker="x", ms=7, color=GRIS, label="valor v1 corregido")]
ax.legend(handles=leyenda, ncol=5, fontsize=9, loc="upper left", bbox_to_anchor=(0, -0.14))
fig.tight_layout()
fig.savefig(DIR_FIG / "porb_por_sector.png", dpi=200, bbox_inches="tight")
plt.show()

## 4) Rotación por sector, relativa a la órbita

Mismo eje x. En el eje y está $P_{\rm rot}/P_{\rm orb}$ de **cada sector**, usando el pico máximo de
Lomb-Scargle del sector. Las líneas horizontales marcan ½, 1 y 2. Un sistema cuyos sectores
saltan entre ½ y 1 (o entre 1 y 2) tiene su pico dominante cambiando de armónico. El periodo
de consenso de la sección 2 (rombo negro) resuelve cuál se repite entre sectores.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
for k, estilo in ((0.5, ":"), (1, "-"), (2, ":")):
    ax.axhline(k, color=TINTA_2 if k == 1 else GRIS, ls=estilo, lw=1, zorder=0)
    ax.text(len(sist) + 0.8, k, {0.5: "½", 1: "1", 2: "2"}[k], va="center", color=TINTA_2, fontsize=10)
for _, r in val.iterrows():
    c = CADENCIA[r["cadencia"]]
    ax.scatter(r["indice"] + DESPLAZ[r["cadencia"]], r["P_rot"] / r["P_orb"], s=45, color=c, alpha=0.9, zorder=2,
               edgecolor="white", lw=0.8)
ax.scatter(sist["indice"], sist["cociente"], marker="D", s=40, facecolor="none", edgecolor=TINTA, lw=1.4, zorder=3)
ax.set_yscale("log"); eje_log(ax, "y")
ax.set_xticks(sist["indice"]); ax.set_xlim(0.3, len(sist) + 0.7)
ax.set_xlabel("Sistema (índice interno)")
ax.set_ylabel(r"$P_{\rm rot}/P_{\rm orb}$")
ax.set_title("Rotación relativa a la órbita: cada sector y el consenso")
ax.grid(axis="x", alpha=0.12)
leyenda = [Line2D([], [], ls="", marker="o", ms=8, color=c, label=k) for k, c in CADENCIA.items()]
leyenda += [Line2D([], [], ls="", marker="D", ms=7, mfc="none", mec=TINTA, label="consenso del sistema")]
ax.legend(handles=leyenda, ncol=4, fontsize=9, loc="upper left", bbox_to_anchor=(0, -0.14))
fig.tight_layout()
fig.savefig(DIR_FIG / "prot_sobre_porb_por_sector.png", dpi=200, bbox_inches="tight")
plt.show()

## 5) Sincronización: $P_{\rm rot}$ vs $P_{\rm orb}$

Un punto por sistema, con los periodos de consenso y sus barras de error. El color es
$|e\cos\omega|$. Los puntos huecos en gris son sistemas sin secundario detectado, donde la
excentricidad no está medida. Los cuadrados marcan sistemas con rotación `ambigua` (el
periodo elegido no aparece en todos los sectores). Cada punto lleva su índice.

Línea continua: $P_{\rm rot}=P_{\rm orb}$ (sincronizado). Punteadas: $P_{\rm rot}=P_{\rm orb}/2$ y
$2P_{\rm orb}$. La barra de color usa una escala de raíz cuadrada, porque casi todos los sistemas
tienen $|e\cos\omega|<0.015$ y uno solo llega a ~0.1.

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 6.6))
lo = 0.8 * np.nanmin([sist["P_orb"].min(), sist["P_rot"].min()])
hi = 1.25 * np.nanmax([sist["P_orb"].max(), sist["P_rot"].max()])
xx = np.array([lo, hi])
ax.plot(xx, xx, color=TINTA_2, lw=1.2, zorder=0)
ax.plot(xx, xx / 2, color=GRIS, lw=1, ls=":", zorder=0)
ax.plot(xx, xx * 2, color=GRIS, lw=1, ls=":", zorder=0)

# raíz cuadrada: la mayoría tiene |e cos w| < 0.015 y uno ~0.1; así se distinguen los pequeños
vmax = np.nanmax(sist["abs_ecosw"]) if sist["abs_ecosw"].notna().any() else 0.1
norm = PowerNorm(gamma=0.5, vmin=0, vmax=max(vmax, 0.02))


for _, r in sist.iterrows():
    if not np.isfinite(r["P_rot"]):
        continue
    con_e = np.isfinite(r["abs_ecosw"])
    color = RAMPA_E(norm(r["abs_ecosw"])) if con_e else "white"
    ax.errorbar(r["P_orb"], r["P_rot"], xerr=r["P_orb_err"] if np.isfinite(r["P_orb_err"]) else None,
                yerr=r["P_rot_err"] if np.isfinite(r["P_rot_err"]) else None,
                fmt="none", ecolor=GRIS, elinewidth=1, capsize=0, zorder=1)
    ax.scatter(r["P_orb"], r["P_rot"], s=85, marker="s" if r["ambiguo"] else "o", zorder=3,
               facecolor=color, edgecolor=TINTA_2 if con_e else GRIS, lw=1.2)

ax.set_xscale("log"); ax.set_yscale("log"); eje_log(ax)
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
ok = sist["P_rot"].notna()
etiquetar(ax, sist.loc[ok, "P_orb"].values, sist.loc[ok, "P_rot"].values, sist.loc[ok, "indice"].astype(str).values)
ax.set_xlabel(r"$P_{\rm orb}$ (d)"); ax.set_ylabel(r"$P_{\rm rot}$ (d, consenso)")
ax.set_title("Rotación vs órbita (un punto por sistema)")
ax.grid(which="major", alpha=0.2)
sm = plt.cm.ScalarMappable(norm=norm, cmap=RAMPA_E); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.02, shrink=0.85)
cb.set_label(r"$|e\cos\omega|$  (cota inferior de $e$; escala raíz)")
cb.outline.set_visible(False)
leyenda = [Line2D([], [], color=TINTA_2, lw=1.2, label=r"$P_{\rm rot}=P_{\rm orb}$ (sincronizado)"),
           Line2D([], [], color=GRIS, lw=1, ls=":", label=r"$P_{\rm rot}=P_{\rm orb}/2$ y $2P_{\rm orb}$"),
           Line2D([], [], ls="", marker="o", ms=9, mfc=RAMPA_E(0.6), mec=TINTA_2, label="rotación consistente"),
           Line2D([], [], ls="", marker="s", ms=9, mfc=RAMPA_E(0.6), mec=TINTA_2, label="rotación ambigua"),
           Line2D([], [], ls="", marker="o", ms=9, mfc="white", mec=GRIS, label="sin secundario (e sin medir)")]
ax.legend(handles=leyenda, fontsize=9, loc="upper left")
fig.tight_layout()
fig.savefig(DIR_FIG / "prot_vs_porb.png", dpi=200, bbox_inches="tight")
plt.show()

## 6) Distribuciones

Histogramas con bines logarítmicos (los periodos abarcan más de un orden de magnitud). En el
cociente $P_{\rm rot}/P_{\rm orb}$ se marcan ½, 1 y 2. Un sistema cuenta una sola vez, con sus
valores de consenso.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
bins_P = np.geomspace(0.2, 15, 16)
axes[0].hist(sist["P_orb"].dropna(), bins=bins_P, color="#2a78d6", edgecolor="white", lw=2)
axes[0].set(xscale="log", xlabel=r"$P_{\rm orb}$ (d)", ylabel="sistemas", title="Periodo orbital")
axes[1].hist(sist["P_rot"].dropna(), bins=bins_P, color="#2a78d6", edgecolor="white", lw=2)
axes[1].set(xscale="log", xlabel=r"$P_{\rm rot}$ (d, consenso)", title="Periodo de rotación")
bins_q = np.geomspace(0.05, 20, 18)
axes[2].hist(sist["cociente"].dropna(), bins=bins_q, color="#2a78d6", edgecolor="white", lw=2)
for k in (0.5, 1, 2):
    axes[2].axvline(k, color=TINTA_2 if k == 1 else GRIS, ls="-" if k == 1 else ":", lw=1)
axes[2].set(xscale="log", xlabel=r"$P_{\rm rot}/P_{\rm orb}$", title="Rotación / órbita")
for a in axes:
    eje_log(a, "x")
    a.yaxis.set_major_locator(MaxNLocator(integer=True))
    a.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(DIR_FIG / "distribuciones_periodos.png", dpi=200, bbox_inches="tight")
plt.show()
sist["cociente"].describe().round(3)

## 7) Profundidad primaria vs secundaria

Profundidades v2 (fondo del eclipse), una marca por sector válido, coloreada por cadencia. El
índice del sistema va junto a la mediana de sus sectores. Los puntos cerca de la diagonal tienen eclipses de profundidad parecida,
es decir, estrellas de brillo superficial parecido.

In [ ]:
d = val.dropna(subset=["Depth_primary", "Depth_secondary"])
d = d.loc[d["Depth_secondary"] > 0]
fig, ax = plt.subplots(figsize=(7, 6.2))
lim = [3e-4, 0.6]
ax.plot(lim, lim, color=GRIS, lw=1, ls=":")
for _, r in d.iterrows():
    ax.scatter(r["Depth_primary"], r["Depth_secondary"], s=50, color=CADENCIA[r["cadencia"]],
               edgecolor="white", lw=0.8, zorder=2, alpha=0.9 if r["secundario_detectado"] else 0.35)
ax.set_xscale("log"); ax.set_yscale("log"); eje_log(ax)
ax.set_xlim(0.012, 0.6); ax.set_ylim(3e-4, 0.6)
med_sis = d.groupby("indice")[["Depth_primary", "Depth_secondary"]].median()
etiquetar(ax, med_sis["Depth_primary"].values, med_sis["Depth_secondary"].values,
          [str(int(i)) for i in med_sis.index])
ax.set_xlabel("Profundidad primaria"); ax.set_ylabel("Profundidad secundaria")
ax.set_title("Profundidades de eclipse (v2)")
ax.grid(which="major", alpha=0.2)
leyenda = [Line2D([], [], ls="", marker="o", ms=8, color=c, label=k) for k, c in CADENCIA.items()]
leyenda.append(Line2D([], [], ls="", marker="o", ms=8, color=GRIS, alpha=0.4, label="secundario no detectado"))
ax.legend(handles=leyenda, fontsize=9, loc="upper left")
fig.tight_layout()
fig.savefig(DIR_FIG / "profundidades.png", dpi=200, bbox_inches="tight")
plt.show()

## 8) Consistencia del periodo orbital entre sectores

`cv_pct` = desviación estándar / media de $P_{\rm orb}$ entre sectores, en %. Después de las
correcciones de `1b` debería ser pequeño para todos los sistemas, salvo los marcados `revisar`.

In [ ]:
multi = val.groupby("TIC").filter(lambda g: g["Sector"].nunique() > 1)
consist = multi.groupby("TIC")["P_orb"].agg(["mean", "std", "count"])
consist["cv_pct"] = 100 * consist["std"] / consist["mean"]
consist.insert(0, "indice", consist.index.map(IDX))
consist.sort_values("cv_pct", ascending=False).round(4)

## 9) Edades

Las edades salen del ajuste massage de la primaria en el diagrama HR (`catalogo_completo_mist_sommers.csv`).
Se usa **MIST** como edad principal porque existe para todos los sistemas; Somers17 (con 17 % de
manchas) solo existe para una parte y se muestra como comparación.

**Barras de error asimétricas.** El catálogo da la media, la mediana y la desviación estándar de la
distribución de edad, pero no sus percentiles. Esas distribuciones son asimétricas (media ≫ mediana),
así que se aproximan con una **lognormal**, para la cual

$$\frac{\text{media}}{\text{mediana}}=e^{\sigma_{\ln}^2/2}\;\Longrightarrow\;\sigma_{\ln}=\sqrt{2\ln(\text{media}/\text{mediana})},$$

y el intervalo del 68 % es $\text{mediana}\times e^{\pm\sigma_{\ln}}$. Si media ≤ mediana, se usa
$\sigma_{\ln}\approx\sigma/\text{mediana}$.

**¿Cuándo la edad significa algo?** La edad por posición en el HR solo discrimina en la
**pre-secuencia principal** (estrellas frías que todavía se contraen). Cerca de la secuencia principal las
isócronas se enciman y la edad queda casi sin restringir. Se marca como **restringida** si

- $T_{\rm eff,A}<6500$ K (primaria con envolvente convectiva, todavía en la PMS en Orión), y
- $\sigma_{\ln}<0.7$ (el intervalo del 68 % abarca menos de un factor 2 en cada lado), y
- edad < 50 Myr. Una edad mayor, para una estrella de Orión, indica que la primaria ya está en
  (o cerca de) la secuencia principal, donde la posición en el HR deja de fechar. Así pasa con
  TIC 11412481 (#4), que da 322 Myr con $T_{\rm eff}=6340$ K.

Las demás se dibujan con un símbolo distinto y **no** entran en las correlaciones.

⚠️ **Sesgo por binariedad.** massage ajusta la luz total del sistema como si fuera una sola estrella. La
luz de la secundaria hace que la primaria parezca más luminosa, y en la PMS eso significa **más joven**.
Por eso las edades de sistemas con secundaria brillante están sesgadas hacia abajo.

In [ ]:
from scipy.stats import spearmanr

cat = pd.read_csv(RUTA_ORION / "catalogo_completo_mist_sommers.csv")
res = pd.read_csv(RUTA_ORION / "resultados_pipeline" / "resumen_por_tic_combinado.csv")


def lognormal(media, mediana, sd):
    if not (np.isfinite(media) and np.isfinite(mediana) and mediana > 0):
        return np.nan
    return np.sqrt(2 * np.log(media / mediana)) if media > mediana * 1.0001 else sd / mediana


ed = cat[["TIC", "age_mean_1", "age_median_1", "age_std_1", "age_median_2", "T_median_1", "lgL_median_1",
          "mass_median_1"]].rename(columns={"age_median_1": "edad", "age_median_2": "edad_somers",
                                           "T_median_1": "Teff_A", "lgL_median_1": "logL_A", "mass_median_1": "M_A"})
ed["sig_ln"] = [lognormal(a, b, c) for a, b, c in zip(ed["age_mean_1"], ed["edad"], ed["age_std_1"])]
ed["edad_lo"] = ed["edad"] * np.exp(-ed["sig_ln"])
ed["edad_hi"] = ed["edad"] * np.exp(ed["sig_ln"])
ed["edad_restringida"] = (ed["Teff_A"] < 6500) & (ed["sig_ln"] < 0.7) & (ed["edad"] < 50)
ed = ed.merge(res[["TIC", "R_A", "R_A_err", "M_B"]], on="TIC", how="left")

cols_ed = ["edad", "edad_lo", "edad_hi", "sig_ln", "edad_somers", "edad_restringida", "Teff_A", "logL_A", "M_A", "R_A",
           "R_A_err", "M_B"]
sist = sist.drop(columns=[c for c in cols_ed if c in sist]).merge(ed[["TIC"] + cols_ed], on="TIC", how="left")
sist["edad_restringida"] = sist["edad_restringida"].fillna(False).astype(bool)

print(f"Edad MIST disponible en {sist['edad'].notna().sum()} de {len(sist)} sistemas; "
      f"restringida en {sist['edad_restringida'].sum()}.")
sist[["indice", "TIC", "Teff_A", "edad", "edad_lo", "edad_hi", "sig_ln", "edad_somers", "edad_restringida"]].round(3)

In [ ]:
# -------- MIST vs Somers17 --------
c = sist.dropna(subset=["edad", "edad_somers"])
fig, ax = plt.subplots(figsize=(6.2, 5.6))
lim = [0.1, 1000]
ax.plot(lim, lim, color=TINTA_2, lw=1)
for _, r in c.iterrows():
    ok = r["edad_restringida"]
    ax.scatter(r["edad"], r["edad_somers"], s=70, marker="o" if ok else "D", zorder=3,
               facecolor="#2a78d6" if ok else "white", edgecolor="#2a78d6", lw=1.4)
ax.set_xscale("log"); ax.set_yscale("log"); eje_log(ax)
ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
etiquetar(ax, c["edad"].values, c["edad_somers"].values, c["indice"].astype(str).values)
ax.set_xlabel("Edad MIST (Myr)"); ax.set_ylabel("Edad Somers17 (Myr)")
ax.set_title("Dependencia de la edad con el modelo")
ax.grid(which="major", alpha=0.2)
ax.legend(handles=[Line2D([], [], ls="", marker="o", ms=8, color="#2a78d6", label="edad restringida"),
                   Line2D([], [], ls="", marker="D", ms=7, mfc="white", mec="#2a78d6", label="poco restringida")],
          fontsize=9, loc="upper left")
fig.tight_layout()
fig.savefig(DIR_FIG / "edades_mist_vs_somers.png", dpi=200, bbox_inches="tight")
plt.show()

## 10) Periodos contra edad

Tres paneles con el mismo eje de edad (MIST, escala log, barras del 68 % lognormal): $P_{\rm orb}$,
$P_{\rm rot}$ (consenso) y $P_{\rm rot}/P_{\rm orb}$. El color es $|e\cos\omega|$, igual que en la sección 5
(blanco = sin secundario detectado). **Círculos**: edad restringida. **Rombos claros**: edad poco
restringida.

En cada panel se da la correlación de rangos de Spearman ($\rho$, con su valor $p$), calculada **solo con
las edades restringidas**. Con tan pocos sistemas, $|\rho|$ tiene que ser grande para ser significativa, así
que los valores $p$ importan más que $\rho$.

**Qué se espera.** Las mareas sincronizan antes de circularizar ($t_{\rm sync}\ll t_{\rm circ}$) y actúan más
rápido con periodos cortos y estrellas grandes, como en la PMS. Si el tiempo importa, los sistemas más
viejos y de periodo corto deberían estar más cerca de $P_{\rm rot}/P_{\rm orb}=1$. La sección 12 lo pone en
números.

In [ ]:
def color_e(v):
    return RAMPA_E(norm(v)) if np.isfinite(v) else "white"


def puntos_edad(ax, y, yerr=None):
    for _, r in sist.dropna(subset=["edad", y]).iterrows():
        ok = r["edad_restringida"]
        xe = [[r["edad"] - r["edad_lo"]], [r["edad_hi"] - r["edad"]]] if np.isfinite(r["sig_ln"]) else None
        ye = r[yerr] if yerr is not None and np.isfinite(r[yerr]) else None
        ax.errorbar(r["edad"], r[y], xerr=xe, yerr=ye, fmt="none", ecolor=GRIS, elinewidth=0.9,
                    alpha=1 if ok else 0.45, zorder=1)
        ax.scatter(r["edad"], r[y], s=70 if ok else 55, marker="o" if ok else "D", zorder=3,
                   facecolor=color_e(r["abs_ecosw"]), edgecolor=TINTA_2 if np.isfinite(r["abs_ecosw"]) else GRIS,
                   lw=1.2, alpha=1 if ok else 0.55)


def rho_texto(y):
    d = sist.loc[sist["edad_restringida"]].dropna(subset=["edad", y])
    if len(d) < 4:
        return "n < 4"
    rho, p = spearmanr(d["edad"], d[y])
    return rf"Spearman $\rho$ = {rho:+.2f}, $p$ = {p:.2f}  (n = {len(d)}, edades restringidas)"


fig, axes = plt.subplots(3, 1, figsize=(9, 11), sharex=True)
for ax, y, yerr, etiqueta in ((axes[0], "P_orb", "P_orb_err", r"$P_{\rm orb}$ (d)"),
                              (axes[1], "P_rot", "P_rot_err", r"$P_{\rm rot}$ (d, consenso)"),
                              (axes[2], "cociente", None, r"$P_{\rm rot}/P_{\rm orb}$")):
    puntos_edad(ax, y, yerr)
    ax.set_yscale("log"); eje_log(ax, "y")
    ax.set_ylabel(etiqueta)
    ax.grid(which="major", alpha=0.2)
    ax.set_title(rho_texto(y), loc="left", fontsize=9, color=TINTA_2)
    d = sist.dropna(subset=["edad", y])
    etiquetar(ax, d["edad"].values, d[y].values, d["indice"].astype(str).values)
for k, estilo in ((0.5, ":"), (1, "-"), (2, ":")):
    axes[2].axhline(k, color=TINTA_2 if k == 1 else GRIS, ls=estilo, lw=1, zorder=0)
axes[2].set_xscale("log"); eje_log(axes[2], "x")
axes[2].set_xlabel("Edad MIST (Myr)")
fig.suptitle("Periodos contra edad", y=0.93)
sm = plt.cm.ScalarMappable(norm=norm, cmap=RAMPA_E); sm.set_array([])
cb = fig.colorbar(sm, ax=axes, pad=0.02, shrink=0.5, location="right")
cb.set_label(r"$|e\cos\omega|$ (escala raíz)"); cb.outline.set_visible(False)
axes[0].legend(handles=[Line2D([], [], ls="", marker="o", ms=8, mfc=RAMPA_E(0.5), mec=TINTA_2, label="edad restringida"),
                        Line2D([], [], ls="", marker="D", ms=7, mfc=RAMPA_E(0.5), mec=TINTA_2, alpha=0.55,
                               label="edad poco restringida"),
                        Line2D([], [], ls="", marker="o", ms=8, mfc="white", mec=GRIS, label="sin secundario")],
               fontsize=9, loc="lower right")
fig.savefig(DIR_FIG / "periodos_vs_edad.png", dpi=200, bbox_inches="tight")
plt.show()

## 11) Diagrama e–P de circularización

Es el diagrama clásico para estudiar circularización (Mathieu & Mazeh 1988; Meibom & Mathieu 2005):
excentricidad contra periodo orbital. Aquí el eje $y$ es $|e\cos\omega|$, la parte robusta y **cota
inferior** de $e$ (sección 12 de los notebooks de starry), y el color es la edad (rampa naranja, log).

La línea vertical marca el **periodo de circularización de binarias PMS**, $P_{\rm circ}\approx7.1$ d
(Meibom & Mathieu 2005). Por debajo de él se esperan órbitas circulares. Los sistemas sin secundario
detectado no tienen $|e\cos\omega|$ medido y aparecen como triángulos en la franja inferior.

Como $|e\cos\omega|\le e$, un punto bajo **no** prueba que la órbita sea circular: puede tener
$\omega\approx90°$. Un punto alto sí prueba que es excéntrica.

**Errores.** Los sistemas con $|e\cos\omega|\ge3\sigma$ se dibujan con su barra de error. Los consistentes
con cero se dibujan como **límite superior al 95 %** (flecha hacia abajo), porque su valor central es
ruido. Los errores son los del bootstrap (§2.10 de `1b`), combinados entre sectores. Son conservadores:
el ajuste completo con starry da errores ~5–10 veces menores cuando converge bien.

In [ ]:
RAMPA_EDAD = LinearSegmentedColormap.from_list("naranja", ["#fde0cf", "#f6a47c", "#eb6834", "#b8461c", "#7a2a0c"])
from matplotlib.colors import LogNorm
norm_edad = LogNorm(vmin=max(0.1, np.nanmin(sist["edad"])), vmax=np.nanmax(sist["edad"]))
PISO = 2e-4   # franja para "sin medir"

fig, ax = plt.subplots(figsize=(8.5, 5.8))
ax.axvline(7.1, color=TINTA_2, lw=1.2, ls="--", zorder=0)
ax.text(7.1 * 0.97, 0.3, r"$P_{\rm circ}\approx7.1$ d" + "\n(PMS, Meibom & Mathieu 2005)", ha="right", va="top",
        fontsize=9, color=TINTA_2)
ax.axhspan(PISO * 0.7, PISO * 1.45, color="#f0efec", zorder=0)
xs, ys, ts = [], [], []
for _, r in sist.iterrows():
    col = RAMPA_EDAD(norm_edad(r["edad"])) if np.isfinite(r["edad"]) else "white"
    y = dibujar_ecosw(ax, r["P_orb"], r, col, PISO, ok=bool(r["edad_restringida"]))
    xs.append(r["P_orb"]); ys.append(y); ts.append(str(int(r["indice"])))
ax.set_xscale("log"); ax.set_yscale("log"); eje_log(ax)
ax.set_ylim(PISO * 0.7, 0.4); ax.set_xlim(0.6, 10)
etiquetar(ax, np.array(xs), np.array(ys), ts)
ax.set_xlabel(r"$P_{\rm orb}$ (d)"); ax.set_ylabel(r"$|e\cos\omega|$  (cota inferior de $e$)")
ax.set_title("Diagrama e–P: circularización")
ax.legend(handles=[Line2D([], [], ls="", marker="o", ms=8, mfc=RAMPA_EDAD(0.5), mec=TINTA_2, label=r"excéntrica ($\geq3\sigma$)"),
                   Line2D([], [], ls="", marker=r"$\downarrow$", ms=10, color=TINTA_2,
                          label=r"consistente con 0: límite al 95 %")], fontsize=8.5, loc="upper left")
ax.text(0.62, PISO, "sin secundario\n(e sin medir)", fontsize=8, color=TINTA_2, va="center")
ax.grid(which="major", alpha=0.2)
sm = plt.cm.ScalarMappable(norm=norm_edad, cmap=RAMPA_EDAD); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.02); cb.set_label("Edad MIST (Myr)"); cb.outline.set_visible(False)
fig.tight_layout()
fig.savefig(DIR_FIG / "diagrama_e_P.png", dpi=200, bbox_inches="tight")
plt.show()

## 12) Tiempos de marea contra edad

La prueba física directa: ¿los sistemas que **ya tuvieron tiempo** de sincronizarse o circularizarse lo
hicieron? Para una primaria con **envolvente convectiva** se usa la marea de equilibrio de Zahn (1977, 1989):

$$\frac{1}{t_{\rm sync}}=6\,\frac{\lambda_2}{t_f}\,q^2\,\frac{MR^2}{I}\left(\frac{R}{a}\right)^6,
\qquad
\frac{1}{t_{\rm circ}}=\frac{21}{2}\,\frac{\lambda_2}{t_f}\,q\,(1+q)\left(\frac{R}{a}\right)^8,$$

con

- $q=M_B/M_A$, y $a$ de la tercera ley de Kepler, $a^3=G(M_A+M_B)P_{\rm orb}^2/4\pi^2$;
- $t_f=(MR^2/L)^{1/3}$, el tiempo de fricción convectiva, que vale ≈ 0.4 años para el Sol (Zahn 1989);
- $\lambda_2\approx0.019\,\alpha^{4/3}\approx0.048$ para una longitud de mezcla $\alpha=2$ (Zahn 1977);
- $I/(MR^2)=0.205$, el de un politropo $n=1.5$, apropiado para estrellas PMS totalmente convectivas.

Como $t_{\rm sync}\propto(a/R)^6$ y $t_{\rm circ}\propto(a/R)^8$, **un error del 10 % en $R$ cambia estos tiempos
en un factor ~1.8–2.1**. Además, la calibración de la marea convectiva es incierta en un factor de unos
pocos. Hay que leerlos como **órdenes de magnitud**. Las barras horizontales combinan en log el error de
la edad (lognormal) y el de $R_A$ ($6\,\sigma_R/R$ o $8\,\sigma_R/R$).

Solo se calculan para primarias frías ($T_{\rm eff,A}<6500$ K). En estrellas calientes con envolvente
radiativa domina la marea dinámica (Zahn 1975), que es mucho más lenta, y esta fórmula no aplica. Se usan
$R_A$ y $L_A$ de massage (Stefan-Boltzmann) y $M_B$ de BHAC15 (`4_pipeline_completo`).

**Lectura.** A la derecha de la línea vertical, $\log(\text{edad}/t)>0$: el sistema ya tuvo tiempo. Si la
teoría es correcta, ahí deberían estar los sincronizados ($P_{\rm rot}/P_{\rm orb}\approx1$) y los circulares
($|e\cos\omega|\approx0$).

In [ ]:
G, MSUN, RSUN, LSUN, DIA, ANIO = 6.674e-11, 1.989e30, 6.957e8, 3.828e26, 86400.0, 3.156e7
LAMBDA2, I_MR2, TF_SOL = 0.019 * 2 ** (4 / 3), 0.205, 0.4   # Zahn (1977, 1989); politropo n = 1.5

z = sist.copy()
z["L_A"] = 10 ** z["logL_A"]
z["q"] = z["M_B"] / z["M_A"]
z["a_Rsun"] = (G * (z["M_A"] + z["M_B"]) * MSUN * (z["P_orb"] * DIA) ** 2 / (4 * np.pi ** 2)) ** (1 / 3) / RSUN
z["a_sobre_R"] = z["a_Rsun"] / z["R_A"]
z["t_f"] = TF_SOL * (z["M_A"] * z["R_A"] ** 2 / z["L_A"]) ** (1 / 3)                    # años
z["t_sync_Myr"] = I_MR2 * z["t_f"] / (6 * LAMBDA2 * z["q"] ** 2) * z["a_sobre_R"] ** 6 / 1e6
z["t_circ_Myr"] = z["t_f"] / (10.5 * LAMBDA2 * z["q"] * (1 + z["q"])) * z["a_sobre_R"] ** 8 / 1e6
z["aplica"] = (z["Teff_A"] < 6500) & z["q"].notna() & z["R_A"].notna()
for t in ("sync", "circ"):
    z[f"log_edad_t{t}"] = np.log10(z["edad"] / z[f"t_{t}_Myr"])
    n = 6 if t == "sync" else 8
    z[f"err_log_{t}"] = np.hypot(z["sig_ln"] / np.log(10), n * z["R_A_err"] / z["R_A"] / np.log(10))

tab_z = z.loc[z["aplica"], ["indice", "TIC", "P_orb", "q", "a_sobre_R", "edad", "t_sync_Myr", "t_circ_Myr",
                            "log_edad_tsync", "log_edad_tcirc", "cociente", "abs_ecosw", "edad_restringida"]]
tab_z.to_csv(DIR_FIG / "tiempos_marea.csv", index=False)
print(f"Tiempos de marea para {len(tab_z)} primarias frías (de {len(z)}). Guardado: {DIR_FIG / 'tiempos_marea.csv'}")
tab_z.round(3)

In [ ]:
d = z.loc[z["aplica"]]
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))

# --- sincronización ---
ax = axes[0]
ax.axvline(0, color=TINTA_2, lw=1.2, ls="--", zorder=0)
for k, estilo in ((0.5, ":"), (1, "-"), (2, ":")):
    ax.axhline(k, color=TINTA_2 if k == 1 else GRIS, ls=estilo, lw=1, zorder=0)
for _, r in d.dropna(subset=["cociente"]).iterrows():
    ok = r["edad_restringida"]
    ax.errorbar(r["log_edad_tsync"], r["cociente"], xerr=r["err_log_sync"], fmt="none", ecolor=GRIS,
                elinewidth=0.9, alpha=1 if ok else 0.45, zorder=1)
    ax.scatter(r["log_edad_tsync"], r["cociente"], s=70 if ok else 55, marker="o" if ok else "D", zorder=3,
               facecolor=color_e(r["abs_ecosw"]), edgecolor=TINTA_2, lw=1.2, alpha=1 if ok else 0.55)
dd = d.dropna(subset=["cociente"])
ax.set_yscale("log"); eje_log(ax, "y")
etiquetar(ax, dd["log_edad_tsync"].values, dd["cociente"].values, dd["indice"].astype(str).values)
ax.set_xlabel(r"$\log_{10}(\mathrm{edad}/t_{\rm sync})$")
ax.set_ylabel(r"$P_{\rm rot}/P_{\rm orb}$")
ax.set_title("Sincronización: ¿hubo tiempo?")
ax.text(0.03, 0.97, "← aún no\n    debería", transform=ax.transAxes, va="top", fontsize=9, color=TINTA_2)
ax.text(0.97, 0.97, "ya debería →\nestar sincronizado", transform=ax.transAxes, va="top", ha="right",
        fontsize=9, color=TINTA_2)
ax.grid(which="major", alpha=0.2)

# --- circularización ---
ax = axes[1]
ax.axvline(0, color=TINTA_2, lw=1.2, ls="--", zorder=0)
ax.axhspan(PISO * 0.7, PISO * 1.45, color="#f0efec", zorder=0)
xs, ys, ts = [], [], []
for _, r in d.iterrows():
    ok = r["edad_restringida"]
    medido = np.isfinite(r["abs_ecosw"])
    y = r["abs_ecosw"] if medido else PISO
    ax.errorbar(r["log_edad_tcirc"], y, xerr=r["err_log_circ"], fmt="none", ecolor=GRIS, elinewidth=0.9,
                alpha=1 if ok else 0.45, zorder=1)
    ax.scatter(r["log_edad_tcirc"], y, s=70 if ok else 55, marker=("o" if ok else "D") if medido else "v", zorder=3,
               facecolor=RAMPA_EDAD(norm_edad(r["edad"])), edgecolor=TINTA_2, lw=1.1, alpha=1 if ok else 0.55)
    xs.append(r["log_edad_tcirc"]); ys.append(y); ts.append(str(int(r["indice"])))
ax.set_yscale("log"); eje_log(ax, "y"); ax.set_ylim(PISO * 0.7, 0.4)
etiquetar(ax, np.array(xs), np.array(ys), ts)
ax.set_xlabel(r"$\log_{10}(\mathrm{edad}/t_{\rm circ})$")
ax.set_ylabel(r"$|e\cos\omega|$")
ax.set_title("Circularización: ¿hubo tiempo?")
ax.text(0.97, 0.97, "ya debería →\nser circular", transform=ax.transAxes, va="top", ha="right", fontsize=9,
        color=TINTA_2)
ax.grid(which="major", alpha=0.2)
sm = plt.cm.ScalarMappable(norm=norm_edad, cmap=RAMPA_EDAD); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=0.02); cb.set_label("Edad MIST (Myr)"); cb.outline.set_visible(False)
fig.tight_layout()
fig.savefig(DIR_FIG / "tiempos_marea_vs_edad.png", dpi=200, bbox_inches="tight")
plt.show()

## 13) Comparación con estudios recientes

Comparación de las binarias frías de Orión con los periodos de corte de circularización y de
sincronización publicados. Se usan solo las primarias frías, con $T_{\rm eff,A}<6250$ K (el cambio de
Kraft, que es también el límite que usan Justesen & Albrecht 2021). Por debajo de él las estrellas
tienen envolvente convectiva, $P_{\rm rot}$ mide rotación por manchas y la marea de equilibrio es la
relevante. Así queda fuera TIC 11412481 (6340 K), cuyo $P_{\rm rot}=\tfrac12P_{\rm orb}$ es casi seguro
variación elipsoidal. También se excluyen los objetos de `EXCLUIR_NO_EB`, que no son binarias
eclipsantes confirmadas.

**Panel izquierdo (circularización).** $|e\cos\omega|$ (cota inferior de $e$) contra $P_{\rm orb}$.
En la regla de abajo, cada fila es el periodo de corte de un estudio:

| Estudio | Muestra | Edad | Corte |
|---|---|---|---|
| Oddo et al. 2025 | binarias de dos enanas M, TESS | campo | ~2 d |
| Zanazzi 2022 | binarias eclipsantes Kepler/TESS | campo | ~3 d y ~6 d |
| Sethi et al. 2026 (EBLM XVII) | F/G/K + M, velocidades radiales | campo | excéntricas solo con $P\gtrsim3$ d |
| Bashi et al. 2023 | binarias espectroscópicas Gaia DR3 | campo | 6.5 d (5700 K) → 2.5 d (6800 K) |
| Torres et al. 2026 | Hyades | ~650 Myr | $5.9\pm1.1$ d |
| Meibom & Mathieu 2005 | binarias PMS | ~1–10 Myr | ~7.1 d |
| **Este trabajo** | **binarias eclipsantes de Orión** | **0.6–9 Myr** | **entre 5.0 d (última circular) y 7.5 d (primera excéntrica)** |

**Panel derecho (sincronización).** $P_{\rm rot}/P_{\rm orb}$ contra $P_{\rm orb}$. La línea discontinua
es la población subsincrónica 7:8 de Hobson-Ritz et al. (2025). En la regla:

| Estudio | Muestra | Edad | Hallazgo |
|---|---|---|---|
| Sethi et al. 2026 (EBLM XVII) | F/G/K + M | campo | "zona de sincronización" con $P<3$ d |
| Hobson-Ritz et al. 2025 | binarias eclipsantes TESS | campo | sincronizadas con $P<10$ d |
| Wang et al. 2026 | Pléyades | 125 Myr | transición a sincronización en 8.6–14 d |
| **Este trabajo** | **Orión** | **0.6–9 Myr** | **sincronizadas hasta 5.0 d** |

**Errores.** En el panel de circularización, los sistemas consistentes con $e\cos\omega=0$ (menos de
$3\sigma$) se dibujan como límites superiores al 95 % (flechas), y los excéntricos con su barra de error.
En el de sincronización, la barra es el error de $P_{\rm rot}/P_{\rm orb}$ propagado del de $P_{\rm rot}$
(consenso entre sectores) y del de $P_{\rm orb}$. La edad aparece solo como color; su incertidumbre
(factor ~2) está en la sección 10.

**Cómo leerla.** Los rangos de "este trabajo" son **límites**: con 12–13 sistemas, el corte real puede
estar en cualquier punto del intervalo marcado. Los valores de la literatura son los reportados por
cada estudio. Verifica en ADS los detalles antes de citarlos, porque varios trabajos de 2026 todavía
pueden cambiar.

In [ ]:
EXCLUIR_NO_EB = [388918012]   # TIC sin eclipses confirmados (modulación por manchas; ver notebook de ese TIC)

TEFF_MAX = 6250   # cambio de Kraft: por encima la envolvente convectiva es muy delgada (Justesen & Albrecht 2021)
fr = sist.loc[(sist["Teff_A"] < TEFF_MAX) & ~sist["TIC"].isin(EXCLUIR_NO_EB)].copy()
circ = fr.dropna(subset=["abs_ecosw"])
sinc = fr.dropna(subset=["cociente"])

# límites de "este trabajo": excéntrica = |e cos w| >= 3 sigma
es_exc = circ["excentrica"].astype(bool)
P_ult_circ = circ.loc[~es_exc, "P_orb"].max()
P_prim_exc = circ.loc[es_exc, "P_orb"].min() if es_exc.any() else np.nan
es_sinc = (sinc["cociente"] - 1).abs() < 0.03
P_ult_sinc = sinc.loc[es_sinc, "P_orb"].max()
print(f"Circularización: última consistente con e cos w = 0 en P = {P_ult_circ:.2f} d, "
      f"primera excéntrica (>= 3 sigma) en P = {P_prim_exc:.2f} d")
print("Límites superiores al 95 % de |e cos w| de las circulares: "
      + ", ".join(f"#{int(r.indice)}: {r.ecosw_lim95:.3f}" for r in circ.loc[~es_exc].itertuples()))
print(f"Sincronización: {es_sinc.sum()}/{len(sinc)} sincronizadas, la de mayor periodo P = {P_ult_sinc:.2f} d")

# filas de las reglas: (etiqueta, tipo, valores)
#   "punto": lista de periodos | "rango": (lo, hi) | "hasta": límite superior | "desde": límite inferior
CIRC_LIT = [
    ("Oddo+ 2025 · enanas M, campo", "punto", [2.0]),
    ("Zanazzi 2022 · Kepler/TESS, campo", "punto", [3.0, 6.0]),
    ("Sethi+ 2026 · EBLM, campo", "desde", 3.0),
    ("Bashi+ 2023 · Gaia, 6800→5700 K", "rango", (2.5, 6.5)),
    ("Torres+ 2026 · Hyades, 650 Myr", "rango", (5.9 - 1.1, 5.9 + 1.1)),
    ("Meibom & Mathieu 2005 · PMS", "punto", [7.1]),
    ("Este trabajo · Orión, 0.6–9 Myr", "rango", (P_ult_circ, P_prim_exc)),
]
SINC_LIT = [
    ("Sethi+ 2026 · EBLM, campo", "hasta", 3.0),
    ("Hobson-Ritz+ 2025 · TESS, campo", "hasta", 10.0),
    ("Wang+ 2026 · Pléyades, 125 Myr", "rango", (8.6, 14.0)),
    ("Este trabajo · Orión, 0.6–9 Myr", "hasta", P_ult_sinc),
]
XMIN, XMAX = 0.6, 16.0
AZUL_EST = "#2a78d6"


def regla(ax, filas):
    for j, (etq, tipo, val) in enumerate(filas[::-1]):
        propio = etq.startswith("Este trabajo")
        col = AZUL_EST if propio else TINTA_2
        j = j - 0.12   # la barra va un poco debajo de su etiqueta
        if tipo == "punto":
            ax.scatter(val, [j] * len(val), marker="|", s=160, lw=2.5, color=col, zorder=3)
        elif tipo == "rango":
            lo, hi = val
            ax.plot([lo, hi], [j, j], color=col, lw=6 if propio else 4, alpha=0.85, solid_capstyle="butt", zorder=2)
        elif tipo == "hasta":
            ax.plot([XMIN, val], [j, j], color=col, lw=4, alpha=0.35, solid_capstyle="butt", zorder=2)
            ax.scatter([val], [j], marker="|", s=160, lw=2.5, color=col, zorder=3)
        elif tipo == "desde":
            ax.plot([val, XMAX], [j, j], color=col, lw=4, alpha=0.35, solid_capstyle="butt", zorder=2)
            ax.scatter([val], [j], marker="|", s=160, lw=2.5, color=col, zorder=3)
    for j, (etq, *_) in enumerate(filas[::-1]):
        propio = etq.startswith("Este trabajo")
        ax.text(XMIN * 1.04, j + 0.3, etq, fontsize=8.5, va="center", ha="left",
                color=AZUL_EST if propio else TINTA_2, fontweight="bold" if propio else "normal")
    ax.set_yticks([])
    ax.set_ylim(-0.5, len(filas) - 0.15)
    ax.grid(axis="x", which="major", alpha=0.2)
    ax.spines["left"].set_visible(False)
    ax.tick_params(axis="y", length=0)


def puntos(ax, d, y):
    ys = []
    for _, r in d.iterrows():
        ok = bool(r["edad_restringida"])
        col = RAMPA_EDAD(norm_edad(r["edad"])) if np.isfinite(r["edad"]) else "white"
        if y == "abs_ecosw":
            ys.append(dibujar_ecosw(ax, r["P_orb"], r, col, 3e-4, ok=ok, z=4, s=80))
            continue
        # cociente P_rot/P_orb con su error propagado
        rel = np.hypot(np.nan_to_num(r["P_rot_err"] / r["P_rot"]), np.nan_to_num(r["P_orb_err"] / r["P_orb"]))
        ax.errorbar(r["P_orb"], r[y], yerr=r[y] * rel, fmt="none", ecolor=GRIS, elinewidth=1, zorder=3)
        ax.scatter(r["P_orb"], r[y], s=80 if ok else 65, marker="o" if ok else "D", zorder=4,
                   facecolor=col, edgecolor=TINTA_2, lw=1.1, alpha=1 if ok else 0.7)
        ys.append(r[y])
    etiquetar(ax, d["P_orb"].values, np.array(ys), d["indice"].astype(str).values)


fig = plt.figure(figsize=(14, 9))
gs = fig.add_gridspec(2, 3, height_ratios=[3, 2.1], width_ratios=[1, 1, 0.035], hspace=0.06, wspace=0.22)
axA, axB = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
axA2, axB2 = fig.add_subplot(gs[1, 0], sharex=axA), fig.add_subplot(gs[1, 1], sharex=axB)
cax = fig.add_subplot(gs[0, 2])

# --- circularización ---
if np.isfinite(P_prim_exc):
    axA.axvspan(P_ult_circ, P_prim_exc, color=AZUL_EST, alpha=0.08, zorder=0)
axA.set_xscale("log"); axA.set_yscale("log")
axA.set_xlim(XMIN, XMAX); axA.set_ylim(3e-4, 0.4)
puntos(axA, circ, "abs_ecosw")
eje_log(axA)
axA.set_ylabel(r"$|e\cos\omega|$  (cota inferior de $e$)")
axA.set_title("Circularización", loc="left")
axA.grid(which="major", alpha=0.2)
plt.setp(axA.get_xticklabels(), visible=False)
regla(axA2, CIRC_LIT)
axA2.set_xlabel(r"$P_{\rm orb}$ (d)")
eje_log(axA2, "x")

# --- sincronización ---
axB.axhline(1.0, color=TINTA_2, lw=1.2, zorder=1)
axB.axhline(7 / 8, color=TINTA_2, lw=1, ls="--", zorder=1)
axB.text(XMAX * 0.95, 7 / 8 * 0.965, "7:8 (Hobson-Ritz+ 2025)", ha="right", va="top", fontsize=8, color=TINTA_2)
axB.text(XMAX * 0.95, 1.035, "sincronizado", ha="right", va="bottom", fontsize=8, color=TINTA_2)
axB.set_xscale("log"); axB.set_yscale("log")
axB.set_xlim(XMIN, XMAX); axB.set_ylim(0.12, 1.6)
puntos(axB, sinc, "cociente")
eje_log(axB)
axB.set_ylabel(r"$P_{\rm rot}/P_{\rm orb}$")
axB.set_title("Sincronización", loc="left")
axB.grid(which="major", alpha=0.2)
plt.setp(axB.get_xticklabels(), visible=False)
regla(axB2, SINC_LIT)
axB2.set_xlabel(r"$P_{\rm orb}$ (d)")
eje_log(axB2, "x")

sm = plt.cm.ScalarMappable(norm=norm_edad, cmap=RAMPA_EDAD); sm.set_array([])
cb = fig.colorbar(sm, cax=cax); cb.set_label("Edad MIST (Myr)"); cb.outline.set_visible(False)
axA.legend(handles=[Line2D([], [], ls="", marker="o", ms=8, mfc=RAMPA_EDAD(0.5), mec=TINTA_2, label="edad restringida"),
                    Line2D([], [], ls="", marker="D", ms=7, mfc=RAMPA_EDAD(0.5), mec=TINTA_2, alpha=0.7,
                           label="edad poco restringida"),
                    Line2D([], [], ls="", marker=r"$\downarrow$", ms=10, color=TINTA_2,
                           label=r"$|e\cos\omega|$ consistente con 0 (límite 95 %)")], fontsize=8.5, loc="upper left")
fig.suptitle(f"Binarias frías de Orión (Teff < {TEFF_MAX} K) frente a estudios recientes", x=0.125, ha="left", y=0.95)
fig.savefig(DIR_FIG / "comparacion_literatura.png", dpi=200, bbox_inches="tight")
fig.savefig(DIR_FIG / "comparacion_literatura.pdf", bbox_inches="tight")
plt.show()